In [ ]:
import sys
sys.path.insert(1, '/home/larnalan/.local/lib/python3.10/site-packages/')
# sys.path.insert(1, '/home/leo/.local/lib/python3.10/site-packages')
# Install dependencies
# check if already installed
if not 'pandas' in sys.modules:
    !pip install pandas
if not 'plotly' in sys.modules:
    !pip install plotly
if not 'kaleido' in sys.modules:
    !pip install -U kaleido
# if not 'ipykernel' in sys.modules:
# !pip install ipykernel
# !pip install --upgrade nbformat


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import visualization_tools as vt
import toolbox as tb
import plotly.graph_objects as go
import os
import json

sns.set_theme()

In [ ]:
dataframeDir = "./website_dataframes/"

EXPERIMENTATION = "PCPNet"
# EXPERIMENTATION = "DGTal"
# EXPERIMENTATION = "CAD"
# EXPERIMENTATION = "Numerical"

From the code: # nbPoints radius noise-position noise-normal {avg,max,variance} for [nbNeighbors,mean,gauss,k1,k2,d1,d2,timings, pos, iShape, normal] non_stable_ratio and variant-name\n";

In [ ]:
def stats_implicit() :
    stats_dir_implicit_double = "../results/implicit/stats/"
    stats_dir_implicit_double_outlier = "../results/implicit_flip/stats/"
    stats_dir_implicit_double_flip = "../results/implicit_outlier/stats/"
    stats_dir_implicit_double_helios = "../results/implicit_helios/stats/"

    stats_dir_implicit_float = "../results/implicit_float/stats/"

    data_list = []
    data_list.append(tb.open_data_all(stats_dir_implicit_double, added_param=[["expe", "DGTal"], ["type", "double"], ["dataset", "implicit"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_implicit_double_outlier, added_param=[["expe", "DGTal"],["type", "double"], ["dataset", "outlier"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_implicit_double_flip, added_param=[["expe", "DGTal"],["type", "double"], ["dataset", "flip"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_implicit_double_helios, added_param=[["expe", "DGTal"],["type", "double"], ["dataset", "helios"]], recursive=False))
    # data_list.append(tb.open_data_all(stats_dir_implicit_float, added_param=[["expe", "DGTal"],["type", "float"], ["dataset", "implicit"]], recursive=False))

    data = pd.concat(data_list)
    data = data.reset_index(drop=True)

    return data

def stats_CAD() :
    stats_dir_CAD = "../results/CAD/stats/"
    stats_dir_CAD_helios = "../results/CAD_Helios/stats/"
    
    stats_dir_CAD_kNNgraph = "../results/CAD_knngraph/stats/"
    stats_dir_CAD_helios_kNNgraph = "../results/CAD_Helios_knngraph/stats/"
    
    data_list=[]
    data_list.append(tb.open_data_all(stats_dir_CAD, added_param=[["expe", "CAD"], ["type", "double"], ["dataset", "CAD"], ["struct", "kD-Tree"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_CAD_helios, added_param=[["expe", "CAD"],["type", "double"], ["dataset", "helios"], ["struct", "kD-Tree"]], recursive=False))
    
    data_list.append(tb.open_data_all(stats_dir_CAD_kNNgraph, added_param=[["expe", "CAD"],["type", "double"], ["dataset", "CAD"], ["struct", "kNN-graph"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_CAD_helios_kNNgraph, added_param=[["expe", "CAD"],["type", "double"], ["dataset", "helios"],["struct", "kNN-graph"]], recursive=False))
    data = pd.concat(data_list)
    data = data.reset_index(drop=True)
    return data

def stats_PCPNet():
    stats_dir_PCPNet_nonoise = "../results/PCPNet/stats/testset_no_noise/"
    stats_dir_PCPNet_lownoise = "../results/PCPNet/stats/testset_low_noise/"
    stats_dir_PCPNet_mednoise = "../results/PCPNet/stats/testset_med_noise/"
    stats_dir_PCPNet_highnoise = "../results/PCPNet/stats/testset_high_noise/"
    stats_dir_PCPNet_gradient = "../results/PCPNet/stats/testset_vardensity_gradient/"
    stats_dir_PCPNet_striped = "../results/PCPNet/stats/testset_vardensity_striped/"
    
    data_list=[]
    data_list.append(tb.open_data_all(stats_dir_PCPNet_nonoise, added_param=[["expe", "PCPNet"], ["type", "double"], ["noise_type", "no"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_PCPNet_lownoise, added_param=[["expe", "PCPNet"],["type", "double"], ["noise_type", "low"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_PCPNet_mednoise, added_param=[["expe", "PCPNet"],["type", "double"], ["noise_type", "med"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_PCPNet_highnoise, added_param=[["expe", "PCPNet"],["type", "double"], ["noise_type", "high"]], recursive=False))

    data_list.append(tb.open_data_all(stats_dir_PCPNet_gradient, added_param=[["expe", "PCPNet"],["type", "double"], ["noise_type", "gradient"]], recursive=False))
    data_list.append(tb.open_data_all(stats_dir_PCPNet_striped, added_param=[["expe", "PCPNet"],["type", "double"], ["noise_type", "striped"]], recursive=False))
    
    data = pd.concat(data_list)
    data = data.reset_index(drop=True)

    data["Noise position"] = -1
    data.loc[data["noise_type"]=="no", "Noise position"] = 0
    data.loc[data["noise_type"]=="low", "Noise position"] = 1
    data.loc[data["noise_type"]=="med", "Noise position"] = 2
    data.loc[data["noise_type"]=="high", "Noise position"] = 3

    # rename Radius column to kNN
    data = data.rename(columns={"Radius": "kNN"})
    data = data[data["kNN"]>10]

    return data

In [ ]:
y_labels_to_names = {
    'N': 'N',
    'Nb neighbors (mean)': 'Neighbors count',
    'Mean curvature (mean)': 'RMSE Mean curvature',
    'Gaussian curvature (mean)': 'RMSE Gaussian curvature',
    'K1 mean': 'RMSE Min curvature',
    'K2 mean': 'RMSE Max curvature',
    'D1 mean': 'RMSAE Min direction',
    'D2 mean': 'RMSAE Max direction',
    'pos mean': 'RMSE Position',
    'iShape mean': 'RMSE Shape Index',
    'normal mean': 'RMSAE Normal',
    'Timings (mean)': 'Timings (ms)',
    'non_stable_ratio': 'Non-stable ratio',
}

In [ ]:
if EXPERIMENTATION == "PCPNet" :
    data_all = stats_PCPNet()
elif EXPERIMENTATION == "DGTal" or EXPERIMENTATION == "Numerical" :
    data_all = stats_implicit()
elif EXPERIMENTATION == "CAD" :
    data_all = stats_CAD()
else :
    print("EXPERIMENTATION not found")
    exit()

data_all = data_all[~data_all.isin([np.inf, -np.inf]).any(axis=1)]

for i, noise_p in enumerate (data_all["Noise position"].unique()) :
    data_all.loc[data_all["Noise position"]==noise_p, "Noise position class"] = i
for j, noise_n in enumerate(data_all["Noise normal"].unique()) :
    data_all.loc[data_all["Noise normal"]==noise_n, "Noise normal class"] = j

for k in range(len(data_all["Noise position"].unique())) :
    # put noise class to k when noise position class is k and noise normal class is k
    data_all.loc[(data_all["Noise position class"]==k) & (data_all["Noise normal class"]==k), "Noise class"] = k
# other, put noise class to -1
data_all.loc[data_all["Noise class"].isna(), "Noise class"] = -1

# Timings (mean) from nanosecond to millisecond
data_all["Timings (mean)"] = data_all["Timings (mean)"] / 1e6

data_all_methods = data_all.copy()

all_methods, methods = tb.methods(data_all, True)

data_all = data_all[data_all["Method"].isin(all_methods)]

color_map_method = {}
for method in all_methods:
    color_map_method[method] = plt.cm.get_cmap("tab20")(all_methods.index(method))

# map the colormap but as rgba value such as 'rgba(x, x, x, 1)' with x between 0 and 256
color_map_to_value = {} 
color_map_to_value_soft = {} 
for method in color_map_method.keys() :
    color_map_to_value[method] = "rgba(" + str(int(color_map_method[method][0]*255)) + ", " + str(int(color_map_method[method][1]*255)) + ", " + str(int(color_map_method[method][2]*255)) + ", 1)"
    color_map_to_value_soft[method] = "rgba(" + str(int(color_map_method[method][0]*255)) + ", " + str(int(color_map_method[method][1]*255)) + ", " + str(int(color_map_method[method][2]*255)) + ", 0.2)"


In [ ]:
properties = {
    "Mean curvature (mean)": tb.mean_curvature_estimation,
    "K1 mean": tb.principal_curvature_estimation,
    "K2 mean": tb.principal_curvature_estimation,
    "Timings (mean)": all_methods,
    "D1 mean": tb.curvature_direction,
    "D2 mean": tb.curvature_direction,
    "normal mean": tb.normal_estimation,
    "iShape mean": tb.principal_curvature_estimation
}

In [ ]:
def create_sub_dataframe(data, for_what, constraint, x_label, y_label, divider=None):
    """
    This function will create all the possible plots, to be given to a statique plot function later.
    ie : for each possible x_label, y_label, divider, constraint, it will create (name, x, y) data to be plotted.
    """
    data_out = {}

    data_constrained = data[data[constraint[0]].isin(constraint[1])]

    if y_label in properties:
        data_constrained = data_constrained[data_constrained["Method"].isin(properties[y_label])]

    for what in data_constrained[for_what].unique():
        data_what = data_constrained[data_constrained[for_what]==what]

        
        data_x = [float(x) if isinstance(x, (int, float, np.number)) else x for x in sorted(data_what[x_label].unique())]
        data_x = [x for x in data_x if not pd.isna(x)]
        

        if divider is not None and divider in data_what.columns:
            for dev in data_what[divider].unique():
                data_y = []
                for x in data_x:
                    mean_value = data_what[(data_what[x_label]==x) & (data_what[divider]==dev)][y_label].mean()
                    if not pd.isna(mean_value):
                        data_y.append(float(mean_value))
                    else:
                        data_y.append(None)
                if data_y:
                    data_out[f'{what}_{dev}'] = [data_x, data_y]
        else:
            data_y = []
            for x in data_x:
                mean_value = data_what[data_what[x_label]==x][y_label].mean()
                if not pd.isna(mean_value):
                    data_y.append(float(mean_value))
                else:
                    data_y.append(None)
            
            if data_y:
                data_out[what] = [data_x, data_y]

    return data_out

# Usage example:
# constraint_test = possible_constraints[0]
# test_data = create_sub_dataframe(data_all, "Method", constraint_test, "Radius", "Mean curvature (mean)", "kernel")

In [ ]:
def dataframe_to_tsx_data(dataframe, possible_x_labels, possible_y_labels, possible_dividers, constraints):
    plot_data = {}
    
    # keys = ["Method", "Shape", "N"]
    keys = ["Method"]

    for constraint in constraints:
        constraint_description = constraint[2]
        for for_what in keys:
            for x_label in possible_x_labels:
                for y_label in possible_y_labels:
                    for divider in possible_dividers + [None]:
                        divider_name = f'_{divider}'
                        if divider is None :
                            divider_name = ""
                        elif divider not in dataframe.columns :
                            continue
                        key = f"{constraint_description}_{x_label}_{y_labels_to_names[y_label]}{divider_name}"
                        data = create_sub_dataframe(dataframe, for_what, constraint, x_label, y_label, divider )
                        plot_data[key] = data
    return plot_data

# plot_data = dataframe_to_tsx_data(data_all, possible_x_labels, possible_y_labels, possible_dividers, possible_constraints)
# print (plot_data.keys())


In [ ]:
def generate_types_file(file_name):
    content = """
export interface PlotDataType {
  [method: string]: [(number | string)[], (number | null)[]];
}

export interface PlotDataSet {
  xLabels: string[];
  yLabels: string[];
  dividers: string[];
  constraints: string[];
  data: {
    [key: string]: PlotDataType;
  };
  colorMap: {
    [method: string]: string;
  };
}

export interface PlotDataAccessors {
  getXLabels: () => string[];
  getYLabels: () => string[];
  getDividers: () => string[];
  getConstraints: () => string[];
  getData: () => { [key: string]: PlotDataType };
  getDataForKey: (key: string) => PlotDataType | undefined;
  getSpecificData: (xLabel: string, yLabel: string, divider: string | null, constraint: string | null) => PlotDataType | null;
  getColorMap : () => { [method: string]: string };
}
"""
    file_path = f"{file_name}.ts"
    with open(file_path, "w") as file:
        file.write(content)
    print(f"File {file_path} has been created.")

def generate_tsx_content(data, x_labels, y_labels, dividers, constraints, color_map, file_name):
    y_labels_list = [y_labels_to_names[yl] if yl in y_labels_to_names else yl for yl in y_labels]
    content = f"""
import {{ PlotDataType, PlotDataSet, PlotDataAccessors }} from '../types';

const {file_name}: PlotDataSet = {{
    xLabels: {json.dumps(x_labels)},
    yLabels: {json.dumps(y_labels_list)},
    dividers: {json.dumps(dividers)},
    constraints: {json.dumps(constraints)},
    data: {json.dumps(data)},
    colorMap: {json.dumps(color_map)},
}};

export const getXLabels = (): string[] => {file_name}.xLabels;
export const getYLabels = (): string[] => {file_name}.yLabels;
export const getDividers = (): string[] => {file_name}.dividers;
export const getConstraints = (): string[] => {file_name}.constraints;
export const getData = (): {{ [key: string]: PlotDataType }} => {file_name}.data;
export const getDataForKey = (key: string): PlotDataType | undefined => {file_name}.data[key];

export const getSpecificData = (xLabel: string, yLabel: string, divider: string | null, constraint: string | null): PlotDataType | null => {{
    const xIndex = {file_name}.xLabels.indexOf(xLabel);
    const yIndex = {file_name}.yLabels.indexOf(yLabel);
    
    if (xIndex === -1 || yIndex === -1) {{
        console.error('Invalid xLabel or yLabel');
        return null;
    }}

    let key = `${{constraint || 'overall'}}_${{xLabel}}_${{yLabel}}`;
    if (divider) {{
        key = `${{key}}_${{divider}}`;
    }}

    const dataForKey = {file_name}.data[key];
    if (!dataForKey) {{
        console.error('No data found for the given parameters');
        return null;
    }}

    return dataForKey;
}};

export const getColorMap = (): {{ [method: string]: string }} => {file_name}.colorMap;

const {file_name}Accessors: PlotDataAccessors = {{
    getXLabels,
    getYLabels,
    getDividers,
    getConstraints,
    getData,
    getDataForKey,
    getSpecificData,
    getColorMap,
}};

export default {file_name}Accessors;
"""
    return content

def write_tsx_file(content, file_name):
    file_path = f"{dataframeDir}/{file_name}.tsx"
    os.makedirs(dataframeDir, exist_ok=True)
    with open(file_path, "w") as file:
        file.write(content)
    print(f"File {file_path} has been created.")

def convert_dict_to_tsx(dataframe, x_labels, y_labels, dividers, constraints, file_name):
    contraints_description_list = [f'{constraint[2]}' for constraint in constraints]
    dividers_list = [divider for divider in dividers if divider in dataframe.columns]
    data_dict = dataframe_to_tsx_data(dataframe, x_labels, y_labels, dividers, constraints)
    content = generate_tsx_content(data_dict, x_labels, y_labels, dividers_list, contraints_description_list, color_map_to_value, file_name)
    write_tsx_file(content, file_name)

In [ ]:
generate_types_file(dataframeDir+"types")

In [ ]:
def createCAD_tsx(dataframe):
    # Conditions et choix pour la colonne 'noise & struct=="kNN-graph"'
    data = dataframe.copy()
    data = data[data["expe"] == "CAD"]

    # Create the column 'noise' with the following values:
    data.loc[(data["dataset"] == "helios") & (data["Noise normal"] == 0) & (data["struct"] == "kD-Tree"),"noise"] = "Helios no noise kD-Tree"
    data.loc[(data["dataset"] == "helios") & (data["Noise normal"] != 0) & (data["struct"] == "kD-Tree"),"noise"] = "Helios noise kD-Tree"
    data.loc[(data["dataset"] == "CAD") & (data["Noise normal"] == 0) & (data["struct"] == "kD-Tree"),"noise"] = "CAD no noise kD-Tree"
    data.loc[(data["dataset"] == "CAD") & (data["Noise normal"] != 0) & (data["struct"] == "kD-Tree"),"noise"] = "CAD noise kD-Tree"
    data.loc[(data["dataset"] == "helios") & (data["Noise normal"] == 0) & (data["struct"] == "kNN-graph"),"noise"] = "Helios no noise kNN-graph"
    data.loc[(data["dataset"] == "helios") & (data["Noise normal"] != 0) & (data["struct"] == "kNN-graph"),"noise"] = "Helios noise kNN-graph"
    data.loc[(data["dataset"] == "CAD") & (data["Noise normal"] == 0) & (data["struct"] == "kNN-graph"),"noise"] = "CAD no noise kNN-graph"
    data.loc[(data["dataset"] == "CAD") & (data["Noise normal"] != 0) & (data["struct"] == "kNN-graph"),"noise"] = "CAD noise kNN-graph"

    possible_x_labels=["Radius", "Noise normal", "Shape", "struct"]
    possible_y_labels=["N", 'Nb neighbors (mean)', "normal mean", "Timings (mean)", "non_stable_ratio"]
    possible_dividers=[]
    
    noise_all = data["noise"].unique()

    possible_constraints=[
        ("noise", noise_all, "all"),
        ("noise", ["CAD no noise kD-Tree"], "No noise kD-Tree"),
        ("noise", ["Helios no noise kD-Tree"], "Helios kD-Tree"),
        ("noise", ["CAD noise kD-Tree"], "Noise normal kD-Tree"),
        ("noise", ["Helios noise kD-Tree"], "Helios + noise normal kD-Tree"),
        ("noise", ["CAD no noise kNN-graph"], "No noise kNN-graph"),
        ("noise", ["Helios no noise kNN-graph"], "Helios kNN-graph"),
        ("noise", ["CAD noise kNN-graph"], "Noise normal kNN-graph"),
        ("noise", ["Helios noise kNN-graph"], "Helios + noise normal kNN-graph"),
    ]
    convert_dict_to_tsx(data, possible_x_labels, possible_y_labels, possible_dividers, possible_constraints, "CAD")

In [ ]:
def createImplicit_tsx(dataframe):
    data = dataframe.copy()
    data = data[data["expe"] == "DGTal"]
    data = data[data["type"] == "double"]
    
    data.loc[(data["dataset"] == "flip"), "noise"] = "flip"
    data.loc[ (data["dataset"] == "outlier"), "noise"] = "outlier"
    data.loc[ (data["dataset"] == "implicit") & (data["Noise class"] == 0), "noise"] = "no noise"
    data.loc[ (data["dataset"] == "implicit") & (data["Noise class"] != 0), "noise"] = "noise"
    data.loc[ (data["dataset"] == "implicit") & (data["Noise normal"] == 0), "noise precision"] = "noise position"
    data.loc[ (data["dataset"] == "implicit") & (data["Noise position"] == 0), "noise precision"] = "noise normal"
    data.loc[ (data["dataset"] == "helios") & (data["Noise class"] == 0), "noise"] = "Helios no noise"
    data.loc[ (data["dataset"] == "helios") & (data["Noise class"] != 0), "noise"] = "Helios noise"

    # print (data["noise"].unique())
    possible_x_labels=["Radius", "Noise position", "Noise class", "Noise normal", "flip-normal", "Shape"]
    possible_y_labels=["N", 'Nb neighbors (mean)', 'Mean curvature (mean)', 'Gaussian curvature (mean)', "K1 mean","K2 mean","D1 mean", "D2 mean", "pos mean", "iShape mean", "normal mean", "Timings (mean)", "non_stable_ratio"]
    possible_dividers=[]
    noise_all = data["noise"].unique()

    possible_constraints=[
        ("noise", noise_all, "all"),
        ("noise", ["no noise"], "No noise"),
        ("noise", ["noise"], "Noise"),
        ("noise precision", ["noise position"], "Noise position"),
        ("noise precision", ["noise normal"], "Noise normal"),
        ("noise", ["Helios no noise"], "Helios no noise"),
        ("noise", ["Helios noise"], "Helios normal noise"),
        ("dataset", ["flip"], "Flip"),
        ("dataset", ["outlier"], "Outlier"),
    ]
    convert_dict_to_tsx(data, possible_x_labels, possible_y_labels, possible_dividers, possible_constraints, "DGTal")

In [ ]:
def createNumerical_tsx(dataframe):
    data = dataframe.copy()
    data = data[data["expe"] == "DGTal"]

    data.loc[(data["type"] == "double") & (data["dataset"] == "implicit") & (data["Noise class"] == 0), "double noise"] = "no noise double"
    data.loc[(data["type"] == "double") & (data["dataset"] == "implicit") & (data["Noise class"] != 0), "double noise"] = "noise double"
    data.loc[(data["type"] == "double") & (data["dataset"] == "implicit") & (data["Noise normal"] == 0), "double noise precision"] = "noise position double"
    data.loc[(data["type"] == "double") & (data["dataset"] == "implicit") & (data["Noise position"] == 0), "double noise precision"] = "noise normal double"

    data.loc[(data["type"] == "float") & (data["dataset"] == "implicit") & (data["Noise class"] == 0), "float noise"] = "no noise float"
    data.loc[(data["type"] == "float") & (data["dataset"] == "implicit") & (data["Noise class"] != 0), "float noise"] = "noise float"
    data.loc[(data["type"] == "float") & (data["dataset"] == "implicit") & (data["Noise normal"] == 0), "float noise precision"] = "noise position float"
    data.loc[(data["type"] == "float") & (data["dataset"] == "implicit") & (data["Noise position"] == 0), "float noise precision"] = "noise normal float"

    # data["noise"] = np.select(conditions, choices, default=data["noise"])
    possible_x_labels=["Radius", "Noise position", "Noise class", "Noise normal", "Shape", "type"]
    possible_y_labels=["N", 'Nb neighbors (mean)', 'Mean curvature (mean)', 'Gaussian curvature (mean)', "K1 mean","K2 mean","D1 mean", "D2 mean", "pos mean", "iShape mean", "normal mean", "Timings (mean)", "non_stable_ratio"]
    possible_dividers=[]
    noise_all_double = data["double noise"].unique()
    noise_all_float = data["float noise"].unique()

    possible_constraints=[
        ("double noise", noise_all_double, "all double"),
        ("float noise", noise_all_float, "all float"),
        ("double noise", ["no noise double"], "No noise double"),
        ("double noise", ["noise double"], "Noise double"),
        ("double noise precision", ["noise position double"], "Noise position double"),
        ("double noise precision", ["noise normal double"], "Noise normal double"),
        ("float noise", ["no noise float"], "No noise float"),
        ("float noise", ["noise float"], "Noise float"),
        ("float noise precision", ["noise position float"], "Noise position float"),
        ("float noise precision", ["noise normal float"], "Noise normal float"),
    ]
    convert_dict_to_tsx(data, possible_x_labels, possible_y_labels, possible_dividers, possible_constraints, "DGTal_Numerical")

In [ ]:
def createPCPNet_tsx(dataframe):
    data = dataframe.copy()
    data = data[data["expe"] == "PCPNet"]
    data = data[data["kNN"] >= 0]
    possible_x_labels=["kNN", "noise_type", "Shape"]
    possible_y_labels=["Timings (mean)", "normal mean", "non_stable_ratio"]
    possible_dividers=[]
    
    kNN = data["kNN"].unique()
    all_noise = data["Noise position"].unique()
    only_noise = [1, 2, 3]
    possible_constraints=[
        ("kNN", kNN, "all"),
        ("noise_type", ["no"], "No noise"),
        ("noise_type", ["gradient", "striped"], "Density"),
        ("noise_type", ["low", "med", "high"], "Noise position"),
    ]
    convert_dict_to_tsx(data, possible_x_labels, possible_y_labels, possible_dividers, possible_constraints, "PCPNet")

In [ ]:
if EXPERIMENTATION == "PCPNet" :
    createPCPNet_tsx(data_all)
elif EXPERIMENTATION == "DGTal" :
    createImplicit_tsx(data_all)
elif EXPERIMENTATION == "Numerical" :
    createNumerical_tsx(data_all)
elif EXPERIMENTATION == "CAD" :
    createCAD_tsx(data_all)